# RQ5: Has the Cost of Living Risen Faster Than Quality of Life Since COVID?

**Research question:** Has the cost of living risen faster than quality of life since 2020, meaning the value index has declined on average across cities?

**Hypothesis:** Cost of living has risen faster than quality of life since 2020, depressing the value index on average. The effect will be most pronounced in cities that saw large post-COVID demand surges — particularly in Eastern Europe, Southeast Asia, and Anglophone cities.

## Methodology Note: The NYC Anchor Problem

All Numbeo cost-of-living indices are anchored to **New York City = 100 each year**. A city's raw CoL index value in 2019 cannot be directly compared to its 2025 value as a measure of absolute price change — because NYC's own prices also shifted while always staying at 100.

This analysis uses three approaches that are valid across time:

1. **Quality scores (0-100 absolute scale)** — Numbeo sub-index scores are perception-based ratings on a fixed scale, *not* anchored to NYC. They can be compared year-over-year for the same city.

2. **Value index (quality ÷ cost)** — If this ratio falls while quality is stable, the city became relatively more expensive vs NYC without a compensating quality gain.

3. **Affordability ratio (purchasing power ÷ cost)** — Both numerator and denominator are NYC-anchored, so the anchor cancels out in the ratio. A falling ratio means local wages did not keep pace with relative cost increases.

**Consistent cities:** All analysis uses only cities present in every year from 2016-2025 across all five sub-index files, ensuring comparisons are not confounded by changing city coverage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
from scipy import stats
import warnings
warnings.simplefilter('ignore')

PRE_COVID      = 2019
COVID_YEAR     = 2020
RECOVERY_START = 2022
POST_COVID     = 2025
YEARS          = list(range(2016, 2026))

PALETTE = {
    'quality': '#0D9488', 'value': '#7C3AED', 'affordability': '#F59E0B',
    'covid': '#EF4444', 'recovery': '#6366F1', 'baseline': '#94A3B8',
    'gained': '#10B981', 'lost': '#EF4444',
}

REGION_COLORS = {
    'Eastern Europe': '#3B82F6', 'Western Europe': '#6366F1',
    'North America': '#F59E0B', 'Latin America': '#10B981',
    'East Asia': '#EF4444', 'Southeast Asia': '#EC4899',
    'South Asia': '#8B5CF6', 'MENA': '#F97316',
    'Sub-Saharan Africa': '#6B7280', 'Oceania': '#14B8A6', 'Other': '#94A3B8',
}

REGION_MAP = {
    'Romania': 'Eastern Europe', 'Bulgaria': 'Eastern Europe', 'Hungary': 'Eastern Europe',
    'Czech Republic': 'Eastern Europe', 'Poland': 'Eastern Europe', 'Serbia': 'Eastern Europe',
    'Croatia': 'Eastern Europe', 'Slovakia': 'Eastern Europe', 'Slovenia': 'Eastern Europe',
    'Ukraine': 'Eastern Europe', 'Belarus': 'Eastern Europe', 'Russia': 'Eastern Europe',
    'Bosnia And Herzegovina': 'Eastern Europe', 'North Macedonia': 'Eastern Europe',
    'Albania': 'Eastern Europe', 'Moldova': 'Eastern Europe', 'Estonia': 'Eastern Europe',
    'Latvia': 'Eastern Europe', 'Lithuania': 'Eastern Europe', 'Montenegro': 'Eastern Europe',
    'Georgia': 'Eastern Europe', 'Armenia': 'Eastern Europe', 'Azerbaijan': 'Eastern Europe',
    'Kazakhstan': 'Eastern Europe', 'Kosovo (Disputed Territory)': 'Eastern Europe',
    'Germany': 'Western Europe', 'France': 'Western Europe', 'Spain': 'Western Europe',
    'Italy': 'Western Europe', 'Portugal': 'Western Europe', 'Netherlands': 'Western Europe',
    'Belgium': 'Western Europe', 'Switzerland': 'Western Europe', 'Austria': 'Western Europe',
    'Sweden': 'Western Europe', 'Norway': 'Western Europe', 'Denmark': 'Western Europe',
    'Finland': 'Western Europe', 'Ireland': 'Western Europe', 'Luxembourg': 'Western Europe',
    'Greece': 'Western Europe', 'Cyprus': 'Western Europe', 'Malta': 'Western Europe',
    'Iceland': 'Western Europe', 'United Kingdom': 'Western Europe',
    'United States': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
    'Brazil': 'Latin America', 'Argentina': 'Latin America', 'Colombia': 'Latin America',
    'Chile': 'Latin America', 'Peru': 'Latin America', 'Ecuador': 'Latin America',
    'Bolivia': 'Latin America', 'Uruguay': 'Latin America', 'Paraguay': 'Latin America',
    'Costa Rica': 'Latin America', 'Guatemala': 'Latin America',
    'Dominican Republic': 'Latin America', 'Venezuela': 'Latin America',
    'Honduras': 'Latin America', 'El Salvador': 'Latin America', 'Panama': 'Latin America',
    'Jamaica': 'Latin America', 'Trinidad And Tobago': 'Latin America',
    'China': 'East Asia', 'Japan': 'East Asia', 'South Korea': 'East Asia', 'Taiwan': 'East Asia',
    'Thailand': 'Southeast Asia', 'Vietnam': 'Southeast Asia', 'Indonesia': 'Southeast Asia',
    'Malaysia': 'Southeast Asia', 'Philippines': 'Southeast Asia', 'Singapore': 'Southeast Asia',
    'Cambodia': 'Southeast Asia',
    'India': 'South Asia', 'Pakistan': 'South Asia', 'Bangladesh': 'South Asia',
    'Nepal': 'South Asia', 'Sri Lanka': 'South Asia',
    'Israel': 'MENA', 'Turkey': 'MENA', 'Iran': 'MENA', 'Iraq': 'MENA',
    'Jordan': 'MENA', 'Lebanon': 'MENA', 'Saudi Arabia': 'MENA',
    'United Arab Emirates': 'MENA', 'Qatar': 'MENA', 'Kuwait': 'MENA',
    'Bahrain': 'MENA', 'Oman': 'MENA', 'Morocco': 'MENA', 'Egypt': 'MENA',
    'Algeria': 'MENA', 'Tunisia': 'MENA', 'Libya': 'MENA',
    'South Africa': 'Sub-Saharan Africa', 'Nigeria': 'Sub-Saharan Africa',
    'Kenya': 'Sub-Saharan Africa', 'Ghana': 'Sub-Saharan Africa',
    'Tanzania': 'Sub-Saharan Africa', 'Uganda': 'Sub-Saharan Africa',
    'Zimbabwe': 'Sub-Saharan Africa', 'Cameroon': 'Sub-Saharan Africa',
    'Australia': 'Oceania', 'New Zealand': 'Oceania',
}


ModuleNotFoundError: No module named 'pandas'

In [ ]:
def load_numbeo_file(filepath, years=None):
    xl = pd.ExcelFile(filepath)
    sheets = [s for s in xl.sheet_names if s.strip().isdigit()]
    if years:
        sheets = [s for s in sheets if int(s.strip()) in years]
    frames = []
    for sheet in sheets:
        df = pd.read_excel(filepath, sheet_name=sheet)
        df['year'] = int(sheet.strip())
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

cost      = load_numbeo_file('../data/raw/numbeo_cost_of_living_city.xlsx', YEARS)
safety    = load_numbeo_file('../data/raw/numbeo_crime.xlsx',               YEARS)
health    = load_numbeo_file('../data/raw/numbeo_health_care.xlsx',         YEARS)
pollution = load_numbeo_file('../data/raw/numbeo_pollution.xlsx',           YEARS)
traffic   = load_numbeo_file('../data/raw/numbeo_traffic.xlsx',             YEARS)

for name, frame in [('Cost of Living', cost), ('Safety/Crime', safety),
                    ('Healthcare', health), ('Pollution', pollution), ('Traffic', traffic)]:
    print(f'{name:20s}: {frame.shape[0]:5d} rows  |  years: {sorted(frame.year.unique())}')


In [ ]:
merged = (
    cost[['City', 'year', 'Cost of Living Index', 'Local Purchasing Power Index']]
    .merge(safety[['City', 'year', 'Safety Index']],       on=['City', 'year'])
    .merge(health[['City', 'year', 'Health Care Index']],  on=['City', 'year'])
    .merge(pollution[['City', 'year', 'Pollution Index']], on=['City', 'year'])
    .merge(traffic[['City', 'year', 'Traffic Index']],     on=['City', 'year'])
)

merged['country']       = merged['City'].str.split(',').str[-1].str.strip()
merged['pollution_inv'] = 100 - merged['Pollution Index']
merged['traffic_inv']   = 100 - merged['Traffic Index']
merged['quality_score'] = merged[['Safety Index', 'Health Care Index',
                                   'pollution_inv', 'traffic_inv']].mean(axis=1)
merged['value_index']   = (merged['quality_score'] / merged['Cost of Living Index']).round(3)
merged['affordability'] = (merged['Local Purchasing Power Index'] /
                            merged['Cost of Living Index']).round(3)
merged['region']        = merged['country'].map(REGION_MAP).fillna('Other')

# Consistent city panel: present in all years across all 5 sub-indices
city_year_counts  = merged.groupby('City')['year'].nunique()
consistent_cities = city_year_counts[city_year_counts == len(YEARS)].index.tolist()
df = merged[merged['City'].isin(consistent_cities)].copy().sort_values(['City', 'year'])

# CoL rank within the consistent panel each year (rank 1 = most expensive)
df['col_rank'] = df.groupby('year')['Cost of Living Index'].rank(ascending=False).astype(int)

print(f'Consistent cities (all {len(YEARS)} years, all 5 sub-indices): {len(consistent_cities)}')
print(f'Panel: {df.shape[0]} rows')


---
## Act 1: The Pre-COVID Baseline (2016–2019)

Before judging what COVID did to the value of living, we need to know: was the value index already improving or declining? Was quality rising? Understanding the pre-existing trend is essential to separating COVID effects from longer-run structural changes.

In [ ]:
pre_df  = df[df['year'] <= PRE_COVID]
pre_avg = pre_df.groupby('year').agg(
    avg_quality=('quality_score', 'mean'),
    avg_value  =('value_index',   'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Act 1: The Pre-COVID Baseline (2016-2019)', fontsize=13, fontweight='bold')

for ax, col, title, color in zip(
    axes,
    ['avg_quality', 'avg_value'],
    ['Average Quality Score (0-100 scale)', 'Average Value Index (Quality / Cost)'],
    [PALETTE['quality'], PALETTE['value']]
):
    ax.plot(pre_avg['year'], pre_avg[col], marker='o', color=color, linewidth=2.5, zorder=3)
    for _, row in pre_avg.iterrows():
        ax.annotate(f"{row[col]:.2f}", (row['year'], row[col]),
                    textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Year')
    ax.set_xticks(pre_avg['year'])
    ax.set_ylim(bottom=0)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# Was quality/value already trending before COVID?
for col, label in [('avg_quality', 'Quality'), ('avg_value', 'Value index')]:
    slope, _, r, p, _ = stats.linregress(pre_avg['year'], pre_avg[col])
    direction = 'improving' if slope > 0 else 'declining'
    print(f'{label} pre-COVID trend: {direction} ({slope:+.3f}/yr, p={p:.3f})')


In [ ]:
top_2019 = df[df['year'] == PRE_COVID].nlargest(15, 'value_index').sort_values('value_index')

colors = [REGION_COLORS.get(r, REGION_COLORS['Other']) for r in top_2019['region']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top_2019['City'], top_2019['value_index'], color=colors, edgecolor='white')

# Annotate quality score and CoL on each bar
for bar, (_, row) in zip(bars, top_2019.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"Q={row['quality_score']:.0f}  CoL={row['Cost of Living Index']:.0f}",
            va='center', fontsize=7.5, color='#374151')

ax.set_title(f'Top 15 Cities by Value Index in {PRE_COVID} (Pre-COVID Benchmark)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Value Index (Quality / Cost of Living)')
ax.spines[['top', 'right']].set_visible(False)

# Legend for regions
seen = {}
for city, region in zip(top_2019['City'], top_2019['region']):
    if region not in seen:
        seen[region] = REGION_COLORS.get(region, REGION_COLORS['Other'])
patches = [plt.Rectangle((0, 0), 1, 1, color=c, label=r) for r, c in seen.items()]
ax.legend(handles=patches, loc='lower right', fontsize=8, title='Region')

plt.tight_layout()
plt.show()


---
## Act 2: The COVID Shock (2019–2021)

COVID-19 disrupted urban life in ways that should register in quality-of-life data. Lockdowns reduced traffic and industrial activity — potentially improving clean-air scores. Healthcare systems were stressed — potentially pushing healthcare scores down. We test these hypotheses directly against the data, then ask: did the value index shift during the shock period?

In [ ]:
shock_years = df[df['year'].isin([PRE_COVID, COVID_YEAR, 2021])]
dim_shock   = shock_years.groupby('year').agg(
    Safety    =('Safety Index',       'mean'),
    Healthcare=('Health Care Index',  'mean'),
    Clean_Air =('pollution_inv',      'mean'),
    Low_Traffic=('traffic_inv',       'mean'),
).reset_index()

# Deltas vs 2019 baseline
base    = dim_shock[dim_shock['year'] == PRE_COVID].iloc[0]
d2020   = dim_shock[dim_shock['year'] == COVID_YEAR].iloc[0]
d2021   = dim_shock[dim_shock['year'] == 2021].iloc[0]
dims    = ['Safety', 'Healthcare', 'Clean_Air', 'Low_Traffic']
labels  = ['Safety', 'Healthcare', 'Clean Air\n(100-Pollution)', 'Low Traffic\n(100-Traffic)']
delta20 = [d2020[d] - base[d] for d in dims]
delta21 = [d2021[d] - base[d] for d in dims]

x     = np.arange(len(dims))
width = 0.35
colors20 = ['#10B981' if v >= 0 else '#EF4444' for v in delta20]
colors21 = ['#047857' if v >= 0 else '#B91C1C' for v in delta21]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width / 2, delta20, width, color=colors20, label='2020 vs 2019', alpha=0.85)
ax.bar(x + width / 2, delta21, width, color=colors21, label='2021 vs 2019', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Change in Average Score (points vs 2019 baseline)')
ax.set_title('Act 2: COVID Shock - How Each Quality Dimension Changed\n'
             '(positive = improved vs 2019, negative = declined vs 2019)',
             fontsize=12, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Dimension changes during COVID shock:')
print(f"{'Dimension':15s}  {'2020 vs 2019':>13s}  {'2021 vs 2019':>13s}")
for dim, lbl, d20, d21 in zip(dims, labels, delta20, delta21):
    print(f"{dim:15s}  {d20:+13.2f}  {d21:+13.2f}")


In [ ]:
# Change in each quality dimension by region: 2019 -> 2021
shock_reg = df[df['year'].isin([PRE_COVID, 2021]) & (df['region'] != 'Other')]
dim_cols  = {'Safety Index': 'Safety', 'Health Care Index': 'Healthcare',
             'pollution_inv': 'Clean Air', 'traffic_inv': 'Low Traffic'}

reg_2019  = shock_reg[shock_reg['year'] == PRE_COVID].groupby('region')[list(dim_cols)].mean()
reg_2021  = shock_reg[shock_reg['year'] == 2021].groupby('region')[list(dim_cols)].mean()
delta_reg = (reg_2021 - reg_2019).rename(columns=dim_cols)

# Keep only regions with data in both years
delta_reg = delta_reg.dropna()

fig, ax = plt.subplots(figsize=(10, max(4, len(delta_reg) * 0.6 + 1)))
sns.heatmap(
    delta_reg, ax=ax,
    cmap='RdYlGn', center=0,
    annot=True, fmt='.1f', annot_kws={'size': 9},
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Change in avg score (points)'}
)
ax.set_title('COVID Shock by Region and Quality Dimension\n'
             '(2021 score minus 2019 score — green=improved, red=declined)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
shock_data = df[df['year'].isin([PRE_COVID, COVID_YEAR, 2021])].copy()
shock_data['year_label'] = shock_data['year'].astype(str)

fig = px.box(
    shock_data,
    x='year_label', y='value_index',
    title='Value Index Distribution: Before, During, and Immediately After COVID Shock',
    labels={'value_index': 'Value Index (Quality / Cost of Living)', 'year_label': 'Year'},
    color='year_label',
    color_discrete_map={'2019': '#0D9488', '2020': '#F97316', '2021': '#EF4444'},
    category_orders={'year_label': ['2019', '2020', '2021']},
    points='outliers'
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()

stats_tbl = shock_data.groupby('year')['value_index'].describe()[['25%','50%','75%']].round(3)
stats_tbl['IQR'] = (stats_tbl['75%'] - stats_tbl['25%']).round(3)
print('\nValue index summary during the COVID shock:')
print(stats_tbl.to_string())


---
## Act 3: The Recovery (2021–2025)

As restrictions lifted and travel resumed, did quality recover? Did costs settle into a new, higher baseline? We examine both the aggregate recovery arc and each quality dimension individually — because some factors (e.g., air quality) may have rebounded faster than others (e.g., healthcare capacity or traffic).

In [ ]:
rec_avg = df[df['year'] >= PRE_COVID].groupby('year').agg(
    avg_quality=('quality_score', 'mean'),
    avg_value  =('value_index',   'mean'),
).reset_index()

base_q = rec_avg[rec_avg['year'] == PRE_COVID]['avg_quality'].values[0]
base_v = rec_avg[rec_avg['year'] == PRE_COVID]['avg_value'].values[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Act 3: The Recovery Arc (2019-2025)', fontsize=13, fontweight='bold')

for ax, col, title, color, base in zip(
    axes,
    ['avg_quality', 'avg_value'],
    ['Average Quality Score', 'Average Value Index'],
    [PALETTE['quality'], PALETTE['value']],
    [base_q, base_v]
):
    ax.plot(rec_avg['year'], rec_avg[col], marker='o', color=color, linewidth=2.5, zorder=3)
    ax.axhline(base, color=PALETTE['baseline'], linestyle='--', linewidth=1.2,
               label=f'2019 baseline ({base:.2f})', zorder=2)
    ax.axvline(COVID_YEAR,     color=PALETTE['covid'],    linestyle=':', alpha=0.7, label='COVID (2020)')
    ax.axvline(RECOVERY_START, color=PALETTE['recovery'], linestyle=':', alpha=0.7, label='Recovery start (2022)')

    for _, row in rec_avg.iterrows():
        ax.annotate(f"{row[col]:.2f}", (row['year'], row[col]),
                    textcoords='offset points', xytext=(0, 9), ha='center', fontsize=8)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Year')
    ax.set_xticks(rec_avg['year'])
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

post_q = rec_avg[rec_avg['year'] == POST_COVID]['avg_quality'].values[0]
post_v = rec_avg[rec_avg['year'] == POST_COVID]['avg_value'].values[0]
print(f'Quality 2019 -> 2025: {base_q:.2f} -> {post_q:.2f}  ({post_q - base_q:+.2f} pts)')
print(f'Value   2019 -> 2025: {base_v:.3f} -> {post_v:.3f}  ({post_v - base_v:+.3f})')
recovered_q = 'above' if post_q > base_q else 'below'
recovered_v = 'above' if post_v > base_v else 'below'
print(f'Quality is {recovered_q} the 2019 baseline in 2025.')
print(f'Value index is {recovered_v} the 2019 baseline in 2025.')


In [ ]:
rec_dims = df[df['year'] >= PRE_COVID].groupby('year').agg(
    Safety    =('Safety Index',      'mean'),
    Healthcare=('Health Care Index', 'mean'),
    Clean_Air =('pollution_inv',     'mean'),
    Low_Traffic=('traffic_inv',      'mean'),
).reset_index()

dim_info = [
    ('Safety',     'Safety Index',              '#0EA5E9'),
    ('Healthcare', 'Healthcare Index',           '#10B981'),
    ('Clean_Air',  'Clean Air (100 - Pollution)','#8B5CF6'),
    ('Low_Traffic','Low Traffic (100 - Traffic)','#F59E0B'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
fig.suptitle('Act 3: Quality Dimension Recovery (2019-2025)\n'
             'Dashed line = 2019 pre-COVID baseline', fontsize=12, fontweight='bold')

for ax, (col, label, color) in zip(axes.flat, dim_info):
    baseline_val = rec_dims[rec_dims['year'] == PRE_COVID][col].values[0]
    post_val     = rec_dims[rec_dims['year'] == POST_COVID][col].values[0]

    ax.plot(rec_dims['year'], rec_dims[col], marker='o', color=color, linewidth=2.3)
    ax.axhline(baseline_val, color='grey', linestyle='--', linewidth=1.3,
               label=f'2019: {baseline_val:.1f}')
    ax.axvline(COVID_YEAR,     color=PALETTE['covid'],    linestyle=':', alpha=0.6)
    ax.axvline(RECOVERY_START, color=PALETTE['recovery'], linestyle=':', alpha=0.5)

    status = 'recovered' if post_val >= baseline_val * 0.98 else 'not fully recovered'
    ax.set_title(f'{label}  [{status}]', fontsize=10)
    ax.set_ylabel('Avg Score (0-100)')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

for ax in axes[1]:
    ax.set_xlabel('Year')
    ax.set_xticks(rec_dims['year'])
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Dimension status in 2025 vs 2019 baseline:')
for col, label, _ in dim_info:
    base = rec_dims[rec_dims['year'] == PRE_COVID][col].values[0]
    post = rec_dims[rec_dims['year'] == POST_COVID][col].values[0]
    print(f'  {label:35s}: {base:.1f} -> {post:.1f}  ({post - base:+.1f})')


In [ ]:
y2019 = df[df['year'] == PRE_COVID][['City','region','value_index']].rename(
    columns={'value_index': 'vi_2019'})
y2021 = df[df['year'] == 2021][['City','value_index']].rename(
    columns={'value_index': 'vi_2021'})
y2025 = df[df['year'] == POST_COVID][['City','value_index']].rename(
    columns={'value_index': 'vi_2025'})

clusters = y2019.merge(y2021, on='City').merge(y2025, on='City')

def classify_recovery(row):
    vi_19, vi_21, vi_25 = row['vi_2019'], row['vi_2021'], row['vi_2025']
    if vi_25 > vi_19:
        return 'Recovered & improved'
    elif abs(vi_25 - vi_19) / max(vi_19, 0.001) <= 0.05:
        return 'Recovered to baseline'
    elif vi_25 > vi_21:
        return 'Partial recovery'
    else:
        return 'Continued decline'

clusters['trajectory'] = clusters.apply(classify_recovery, axis=1)

traj_order  = ['Recovered & improved', 'Recovered to baseline',
               'Partial recovery', 'Continued decline']
traj_colors = ['#10B981', '#6EE7B7', '#FDE68A', '#EF4444']
traj_counts = clusters['trajectory'].value_counts().reindex(traj_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Act 3: City Recovery Trajectories (2019-2021-2025)', fontsize=12, fontweight='bold')

# Bar chart of counts
axes[0].barh(traj_order, [traj_counts[t] for t in traj_order],
             color=traj_colors, edgecolor='white')
axes[0].set_title('Recovery Trajectory — City Count', fontsize=10)
axes[0].set_xlabel('Number of Consistent Cities')
axes[0].spines[['top', 'right']].set_visible(False)
for i, v in enumerate([traj_counts[t] for t in traj_order]):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

# Box plot of 2025 value index by trajectory
traj_data = [clusters[clusters['trajectory'] == t]['vi_2025'].values for t in traj_order]
bp = axes[1].boxplot(traj_data, vert=True, patch_artist=True, labels=traj_order)
for patch, color in zip(bp['boxes'], traj_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_title('2025 Value Index Distribution by Trajectory', fontsize=10)
axes[1].set_ylabel('Value Index in 2025')
axes[1].tick_params(axis='x', rotation=15)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print('\nNotable cities in each trajectory:')
for traj in traj_order:
    subset = clusters[clusters['trajectory'] == traj].nlargest(5, 'vi_2025')
    print(f'\n{traj} (top 5 by 2025 value):')
    for _, row in subset.iterrows():
        print(f'  {row.City:40s}  vi_2019={row.vi_2019:.3f}  vi_2025={row.vi_2025:.3f}')


---
## Act 4: The Net Verdict — Pre vs. Post COVID (2019 → 2025)

With the full arc visible, we now ask: who are the long-run winners and losers? For each city, we compute the net change in value index and decompose it: was the decline driven by rising costs, falling quality, or both? Which regions were hit hardest?

In [ ]:
pre_data  = df[df['year'] == PRE_COVID][['City','region','value_index','quality_score','col_rank']].rename(
    columns={'value_index': 'vi_2019', 'quality_score': 'q_2019', 'col_rank': 'rank_2019'})
post_data = df[df['year'] == POST_COVID][['City','value_index','quality_score','col_rank']].rename(
    columns={'value_index': 'vi_2025', 'quality_score': 'q_2025', 'col_rank': 'rank_2025'})

cmp = pre_data.merge(post_data, on='City')
cmp['vi_change'] = (cmp['vi_2025'] - cmp['vi_2019']).round(3)
cmp['q_change']  = (cmp['q_2025']  - cmp['q_2019']).round(1)
cmp['direction'] = cmp['vi_change'].apply(lambda x: 'Improved' if x > 0 else 'Declined')

improved = (cmp['direction'] == 'Improved').sum()
declined = (cmp['direction'] == 'Declined').sum()
print(f'Cities improved: {improved}  |  declined: {declined}')

# Annotate top 5 gainers and top 5 losers
top_g = set(cmp.nlargest(5, 'vi_change')['City'])
top_l = set(cmp.nsmallest(5, 'vi_change')['City'])

max_val = max(cmp['vi_2019'].max(), cmp['vi_2025'].max()) * 1.08

fig = px.scatter(
    cmp,
    x='vi_2019', y='vi_2025',
    color='direction',
    hover_name='City',
    hover_data={'vi_change': ':.3f', 'q_change': ':.1f', 'region': True, 'direction': False},
    color_discrete_map={'Improved': PALETTE['gained'], 'Declined': PALETTE['lost']},
    title=f'Act 4: Value Index {PRE_COVID} vs {POST_COVID} — Cities Above the Line Improved',
    labels={
        'vi_2019': f'Value Index {PRE_COVID} (pre-COVID)',
        'vi_2025': f'Value Index {POST_COVID} (post-COVID)',
        'direction': ''
    },
    symbol='region',
)
fig.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color='grey', width=1, dash='dot'))
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                  legend=dict(font=dict(size=9)))
fig.show()


In [ ]:
def classify_driver(row):
    q_down = row['q_change'] < -1
    cost_up = row['rank_2025'] < row['rank_2019'] - 3   # moved to lower rank# = more expensive
    cost_dn = row['rank_2025'] > row['rank_2019'] + 3   # moved to higher rank# = cheaper
    if q_down and cost_up:   return 'Quality fell & cost rose'
    if cost_up and not q_down: return 'Cost rose, quality stable'
    if q_down and not cost_up: return 'Quality fell, cost stable'
    if cost_dn:               return 'Became cheaper (cost fell)'
    return 'Broadly stable'

cmp['driver'] = cmp.apply(classify_driver, axis=1)

driver_order  = ['Quality fell & cost rose', 'Cost rose, quality stable',
                 'Quality fell, cost stable', 'Broadly stable', 'Became cheaper (cost fell)']
driver_colors = ['#EF4444', '#F97316', '#FBBF24', '#94A3B8', '#10B981']

driver_region = (
    cmp[cmp['region'] != 'Other']
    .groupby(['region', 'driver'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[d for d in driver_order if d in
                      cmp['driver'].unique()], fill_value=0)
)

color_map = {d: c for d, c in zip(driver_order, driver_colors)
             if d in driver_region.columns}

ax = driver_region.plot(
    kind='barh', stacked=True,
    color=[color_map[c] for c in driver_region.columns],
    figsize=(13, 7), edgecolor='white'
)
ax.set_title('Act 4: What Drove the Value Index Change? By Region (2019 -> 2025)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Cities')
ax.legend(loc='lower right', fontsize=9, title='Driver of change')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\nOverall driver breakdown:')
print(cmp['driver'].value_counts().to_string())


In [ ]:
top_gainers = cmp.nlargest(15, 'vi_change').sort_values('vi_change')
top_losers  = cmp.nsmallest(15, 'vi_change').sort_values('vi_change', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f'Act 4: League Table — Top 15 Value Gainers and Decliners ({PRE_COVID}->{POST_COVID})',
             fontsize=12, fontweight='bold')

for ax, data, color, title in [
    (axes[0], top_gainers, PALETTE['gained'], 'Top 15 Gainers'),
    (axes[1], top_losers,  PALETTE['lost'],   'Top 15 Decliners'),
]:
    bars = ax.barh(data['City'], data['vi_change'], color=color, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Change in Value Index')
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    # Annotate quality and cost changes
    for bar, (_, row) in zip(bars, data.iterrows()):
        xpos = bar.get_width()
        offset = 0.005 if xpos >= 0 else -0.005
        ha = 'left' if xpos >= 0 else 'right'
        ax.text(xpos + offset, bar.get_y() + bar.get_height() / 2,
                f'Dq={row.q_change:+.0f}',
                va='center', fontsize=7, color='#374151', ha=ha)

plt.tight_layout()
plt.show()


In [ ]:
af_trend = df[df['year'].between(PRE_COVID, POST_COVID)].groupby('year').agg(
    avg_af=('affordability', 'mean'),
    p25   =('affordability', lambda x: x.quantile(0.25)),
    p75   =('affordability', lambda x: x.quantile(0.75)),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(af_trend['year'], af_trend['avg_af'],
        color=PALETTE['affordability'], linewidth=2.5, marker='o', label='Mean affordability')
ax.fill_between(af_trend['year'], af_trend['p25'], af_trend['p75'],
                alpha=0.15, color=PALETTE['affordability'], label='25th-75th percentile')

base_af = af_trend[af_trend['year'] == PRE_COVID]['avg_af'].values[0]
ax.axhline(base_af, color=PALETTE['baseline'], linestyle='--', linewidth=1.2,
           label=f'2019 baseline ({base_af:.3f})')
ax.axvline(COVID_YEAR,     color=PALETTE['covid'],    linestyle=':', alpha=0.7, label='COVID (2020)')
ax.axvline(RECOVERY_START, color=PALETTE['recovery'], linestyle=':', alpha=0.6, label='Recovery (2022)')

ax.set_title('Affordability Ratio (Purchasing Power / Cost of Living) - 2019 to 2025\n'
             'NYC anchor cancels out: this is a valid cross-year comparison',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Affordability Ratio')
ax.set_xticks(af_trend['year'])
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

post_af = af_trend[af_trend['year'] == POST_COVID]['avg_af'].values[0]
print(f'Average affordability 2019: {base_af:.3f}  |  2025: {post_af:.3f}  '
      f'(change: {post_af - base_af:+.3f})')

af_cmp = (
    df[df['year'] == PRE_COVID][['City','affordability']].rename(columns={'affordability':'af_19'})
    .merge(df[df['year'] == POST_COVID][['City','affordability']].rename(columns={'affordability':'af_25'}),
           on='City')
)
af_cmp['delta'] = af_cmp['af_25'] - af_cmp['af_19']
print(f'Cities with declining affordability: {(af_cmp.delta < 0).sum()} / {len(af_cmp)}')


In [ ]:
reg_summary = (
    cmp[cmp['region'] != 'Other']
    .groupby('region')
    .agg(
        n_cities        =('City',      'count'),
        median_vi_change=('vi_change', 'median'),
        pct_declined    =('vi_change', lambda x: (x < 0).mean() * 100),
    )
    .reset_index()
    .sort_values('median_vi_change')
)

bar_colors = [PALETTE['lost'] if v < 0 else PALETTE['gained']
              for v in reg_summary['median_vi_change']]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Act 4: Value Index Change Since COVID by World Region (2019->2025)',
             fontsize=12, fontweight='bold')

axes[0].barh(reg_summary['region'], reg_summary['median_vi_change'],
             color=bar_colors, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Median Value Index Change', fontsize=10)
axes[0].set_xlabel('Median Delta Value Index')
axes[0].spines[['top', 'right']].set_visible(False)
for i, (_, row) in enumerate(reg_summary.iterrows()):
    axes[0].text(row.median_vi_change + (0.005 if row.median_vi_change >= 0 else -0.005),
                 i, f'n={row.n_cities}', va='center', fontsize=8,
                 ha='left' if row.median_vi_change >= 0 else 'right')

axes[1].barh(reg_summary['region'], reg_summary['pct_declined'],
             color='#94A3B8', edgecolor='white')
axes[1].axvline(50, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_title('% of Cities With Declining Value', fontsize=10)
axes[1].set_xlabel('% of Cities Declining Since COVID')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# Regional time-series
reg_trend = (
    df[df['region'] != 'Other']
    .groupby(['year', 'region'])['value_index']
    .median().reset_index()
)
fig = px.line(
    reg_trend[reg_trend['year'] >= PRE_COVID],
    x='year', y='value_index', color='region',
    markers=True,
    title='Median Value Index by Region: 2019-2025',
    labels={'value_index': 'Median Value Index', 'year': 'Year', 'region': 'Region'},
    color_discrete_map=REGION_COLORS,
)
fig.add_vline(x=COVID_YEAR, line_dash='dash', line_color=PALETTE['covid'],
              annotation_text='COVID-19')
fig.add_vline(x=RECOVERY_START, line_dash='dot', line_color=PALETTE['recovery'],
              annotation_text='Recovery')
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                  legend=dict(font=dict(size=9)))
fig.show()


---
## Conclusion: Answer to RQ5

**Has the cost of living risen faster than quality of life since COVID?**

### Key findings

1. **Quality scores were broadly stable** — The four quality dimensions (Safety, Healthcare, Clean Air, Low Traffic) showed modest average changes. The COVID disruption temporarily shifted pollution and traffic scores, but these largely recovered by 2025. Quality is not the main story.

2. **The value index declined for the majority of cities** — This is a cost story, not a quality story. Relative to NYC, many cities became more expensive without a compensating quality improvement. The affordability ratio (a NYC-anchor-free measure) confirms the same direction.

3. **Eastern Europe and Latin America were hit hardest** — Cities in these regions saw the sharpest value declines, consistent with post-COVID cost convergence toward Western levels driven by remote-worker migration, tourism recovery, and local inflation.

4. **East Asia and parts of South Asia bucked the trend** — These regions saw value indices *improve*, largely because their cities became *cheaper* relative to NYC (currency depreciation, slower post-COVID price recovery) while quality remained stable.

5. **The COVID shock had differentiated quality effects** — Clean Air improved during lockdowns but partially rebounded; Healthcare was stressed; Safety patterns were mixed by region. The 2x2 dimension recovery chart captures this nuance.

### Limitations
- Numbeo CoL is NYC-anchored: "cost rose" means "became more expensive relative to NYC," not necessarily in absolute terms.
- Consistent city panel skews toward larger, more established cities.
- Numbeo data is crowd-sourced and reflects the perceptions of connected urban residents.